In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, welch
import os
import time

# =========================
# INPUT: list of subject files
# =========================
file_list = [
    r'../Raw_Data/EEG_Subject_1.csv',
    r'../Raw_Data/EEG_Subject_2.csv',
    r'../Raw_Data/EEG_Subject_3.csv',
    r'../Raw_Data/EEG_Subject_4.csv',
    r'../Raw_Data/EEG_Subject_5.csv',
    r'../Raw_Data/EEG_Subject_6.csv',
    r'../Raw_Data/EEG_Subject_7.csv',
    r'../Raw_Data/EEG_Subject_8.csv',
    r'../Raw_Data/EEG_Subject_9.csv',
    r'../Raw_Data/EEG_Subject_10.csv',
    r'../Raw_Data/EEG_Subject_11.csv',
    r'../Raw_Data/EEG_Subject_12.csv',
    r'../Raw_Data/EEG_Subject_13.csv',
    r'../Raw_Data/EEG_Subject_14.csv',
    r'../Raw_Data/EEG_Subject_15.csv',
    r'../Raw_Data/EEG_Subject_16.csv',
    r'../Raw_Data/EEG_Subject_17.csv',
    r'../Raw_Data/EEG_Subject_18.csv',
    r'../Raw_Data/EEG_Subject_19.csv',
    r'../Raw_Data/EEG_Subject_20.csv',
    r'../Raw_Data/EEG_Subject_21.csv',
    r'../Raw_Data/EEG_Subject_22.csv',
    r'../Raw_Data/EEG_Subject_23.csv',
    r'../Raw_Data/EEG_Subject_24.csv',
    r'../Raw_Data/EEG_Subject_25.csv',
    r'../Raw_Data/EEG_Subject_26.csv',
    r'../Raw_Data/EEG_Subject_27.csv'
]

# Output folder (will be created if it doesn't exist)
out_dir = r"../PSD_data1"
os.makedirs(out_dir, exist_ok=True)

# Images root folder inside PSD_data
img_root_dir = os.path.join(out_dir, "PSD_img")
os.makedirs(img_root_dir, exist_ok=True)

# =========================
# PARAMETERS
# =========================
fs = 256  # Sampling frequency in Hz
delay_sec = 1
stim_duration_sec = 7
delay_samples = int(fs * delay_sec)
stim_duration_samples = int(fs * stim_duration_sec)
stim_event_ids = [33024, 33025, 33026, 33027]

# =========================
# FILTER DEFINITIONS
# =========================
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    return butter(order, [low, high], btype='band')

def bandpass_filter(data, lowcut, highcut, fs, order=4):
    b, a = butter_bandpass(lowcut, highcut, fs, order)
    return filtfilt(b, a, data)

def notch_filter(data, fs, notch_freq=50.0, quality_factor=30.0):
    b, a = iirnotch(notch_freq, quality_factor, fs)
    return filtfilt(b, a, data)

# Filter parameters
lowcut = 6
highcut = 30
notch_freq = 50.0
quality_factor = 30.0

# =========================
# PSD WINDOWING PARAMETERS
# =========================
W = 512        # 2-second window
S = 64         # step size (0.25 s)
channels = ["PO7", "O1", "O2", "PO8"]
max_bins = 50  # first 50 PSD bins (as in the original code)

# Labels for the event IDs
stim_labels = {
    0: "Standby",
    33024: "No stimulus",
    33025: "Stimulus 10 Hz",
    33026: "Stimulus 12 Hz",
    33027: "Stimulus 8.57 Hz"
}

# =========================
# PLOT: mean PSD for a given event
# =========================
def plot_psd_media(psd_df, freqs, event_id, subject_name, img_dir_subject):
    if event_id not in stim_labels:
        print(f"⚠️ Event Id {event_id} is not valid.")
        return

    stim_df = psd_df[psd_df["Event Id"] == event_id]
    if stim_df.empty:
        print(f"No windows found for Event Id {event_id}")
        return

    plt.figure(figsize=(12, 5))
    for ch in channels:
        cols = [f"{ch}_f{i+1}" for i in range(max_bins)]
        psd_vals = stim_df[cols].mean().values
        plt.plot(freqs[:max_bins], psd_vals, label=ch)

    title = f"[{subject_name}] Mean PSD - {stim_labels[event_id]} (Event Id: {event_id})"
    plt.title(title)
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Power (a.u.)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    # Save the figure in PSD_data/PSD_img/<subject_name>/
    fname = f"PSD_{subject_name}_Event_{event_id}.png"
    save_path = os.path.join(img_dir_subject, fname)
    plt.savefig(save_path, dpi=300)

    # Show the figure
    plt.show()

    # Close figure to avoid accumulating too many open figures
    plt.close()

# =========================
# LOOP OVER SUBJECTS
# =========================
for file_path in file_list:

    subject_name = os.path.splitext(os.path.basename(file_path))[0]
    print("\n==============================")
    print(f"Processing: {subject_name}")
    print(f"File: {file_path}")
    print("==============================")

    # Create subject subfolder ONLY inside PSD_img
    img_dir_subject = os.path.join(img_root_dir, subject_name)
    os.makedirs(img_dir_subject, exist_ok=True)

    # === 1. LOAD AND CLEAN ===
    df = pd.read_csv(file_path)

    # Remove all rows before the experiment start (Event Id = 32769)
    start_exp_idx = df[df["Event Id"] == 32769].index.min()
    if pd.notna(start_exp_idx):
        df = df.loc[start_exp_idx:].reset_index(drop=True)

    # Drop unnecessary columns
    columns_to_drop = [
        "Epoch", "Event Date", "Event Duration",
        "Channel 1", "Channel 2", "Channel 3", "Channel 4",
        "Channel 5", "Channel 6", "Channel 7", "Channel 8"
    ]
    df.drop(columns=columns_to_drop, inplace=True, errors="ignore")

    # Rename the remaining channels
    df.rename(columns={
        "Channel 9": "PO7",
        "Channel 10": "O1",
        "Channel 11": "O2",
        "Channel 12": "PO8"
    }, inplace=True)

    # === 2. EVENT EXTENSION WITH DELAY ===
    df_filled = df.copy()
    stim_indices = df[df["Event Id"].isin(stim_event_ids)].index

    for idx in stim_indices:
        stim_id = df.loc[idx, "Event Id"]
        df_filled.at[idx, "Event Id"] = np.nan  # Remove event from the current row
        start = idx + delay_samples
        end = start + stim_duration_samples

        # Safety check to avoid out-of-bounds indexing
        if start >= len(df_filled):
            continue
        if end > len(df_filled):
            end = len(df_filled)

        # Insert stimulus after 1s for 7s
        df_filled.loc[start:end - 1, "Event Id"] = stim_id

        # Clear the end marker (optional, kept to stay close to original code)
        if end < len(df_filled):
            df_filled.at[end, "Event Id"] = np.nan

    df_filled["Event Id"] = df_filled["Event Id"].fillna(0)

    # === 3. COMPUTE PSD FEATURES (WINDOWED) ===
    psd_features = []
    labels = []
    freqs = []

    for start in range(0, len(df_filled) - W + 1, S):
        end = start + W
        window = df_filled.iloc[start:end]

        # Inizia a misurare il tempo
        start_time = time.perf_counter()
        
        feature_vector = []
        for ch in channels:
            signal = window[ch].values
            bandpassed = bandpass_filter(signal, lowcut, highcut, fs)
            filtered = notch_filter(bandpassed, fs, notch_freq, quality_factor)
            freqs, psd = welch(filtered, fs=fs, nperseg=256, noverlap=192)
            feature_vector.extend(psd[:max_bins])  # 50 PSD values per channel

        # Ferma il cronometro
        end__time = time.perf_counter()
        
        # Calcolo della durata (opzionale, se vuoi stamparlo o salvarlo)
        execution_time = end__time - start_time
        print(f"Elaborazione finestra completata in {execution_time:.4f} secondi")
        
        event_id = window["Event Id"].iloc[-1]
        psd_features.append(feature_vector)
        labels.append(event_id)

    # Feature table
    column_names = [f"{ch}_f{i+1}" for ch in channels for i in range(max_bins)]
    psd_df = pd.DataFrame(psd_features, columns=column_names)
    psd_df["Event Id"] = labels

    # === 4. PLOT MEAN PSD PER EVENT AND SAVE IT IN PSD_img/<subject_name>/ ===
    plot_psd_media(psd_df, freqs, 33027, subject_name, img_dir_subject)  # 8.57 Hz
    plot_psd_media(psd_df, freqs, 33025, subject_name, img_dir_subject)  # 10 Hz
    plot_psd_media(psd_df, freqs, 33026, subject_name, img_dir_subject)  # 12 Hz

    # === 5. SAVE FINAL PSD DATASET CSV (kept in PSD_data root) ===
    out_csv = os.path.join(out_dir, f"PSD_{subject_name}.csv")
    psd_df.to_csv(out_csv, index=False)
    print(f"✅ Saved: {out_csv} | shape={psd_df.shape}")



Processing: EEG_Subject_1
File: ../Raw_Data/EEG_Subject_1.csv
Elaborazione finestra completata in 0.0063 secondi
Elaborazione finestra completata in 0.0042 secondi
Elaborazione finestra completata in 0.0041 secondi
Elaborazione finestra completata in 0.0040 secondi
Elaborazione finestra completata in 0.0040 secondi
Elaborazione finestra completata in 0.0039 secondi
Elaborazione finestra completata in 0.0040 secondi
Elaborazione finestra completata in 0.0039 secondi
Elaborazione finestra completata in 0.0040 secondi
Elaborazione finestra completata in 0.0043 secondi
Elaborazione finestra completata in 0.0039 secondi
Elaborazione finestra completata in 0.0039 secondi
Elaborazione finestra completata in 0.0040 secondi
Elaborazione finestra completata in 0.0039 secondi
Elaborazione finestra completata in 0.0039 secondi
Elaborazione finestra completata in 0.0039 secondi
Elaborazione finestra completata in 0.0039 secondi
Elaborazione finestra completata in 0.0039 secondi
Elaborazione finest

KeyboardInterrupt: 